# WaveSlide - Data Preparation - EDA

### Collaborators:
- Raul Andres Cerreño Zeballos
- Leicy Cristel Cahuana Lopez
- Diego Fabrizio Mucha Alvarez

## 1. Load and Clean Data


### 1.1. Import libraries

In [1]:
import os
import json
from pathlib import Path
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import numpy as np

### 1.2. Define Loading and cleaning function

In [7]:
from pathlib import Path
import json
import cv2
import numpy as np
import pandas as pd


# -------------------------------
# 1. Load annotations from raw data
# -------------------------------

def load_data(images_root_path, annotations_root_path):
    
    images_root_path = Path(images_root_path)
    annotations_root_path = Path(annotations_root_path)

    df_rows = []

    for annotation_file in annotations_root_path.glob("*.json"):

        class_name = annotation_file.stem
        image_dir = images_root_path / class_name

        with open(annotation_file, "r") as f:
            annotations = json.load(f)

        for image_id, ann in annotations.items():

            image_path = image_dir / f"{image_id}.jpg"

            for bbox, label in zip(ann["bboxes"], ann["labels"]):

                df_rows.append({
                    "image_id": image_id,
                    "image_path": str(image_path),
                    "image_exists": image_path.exists(),
                    "label": label,
                    "bbox_x": bbox[0],
                    "bbox_y": bbox[1],
                    "bbox_w": bbox[2],
                    "bbox_h": bbox[3],
                })

    df = pd.DataFrame(df_rows)

    # Keep only existing images
    df = df[df["image_exists"] == True].drop(columns=["image_exists"])

    # Remove images with multiple bboxes if you only want one hand per image
    df = df[~df["image_id"].duplicated(keep=False)]

    # Remove no_gesture
    df = df[df["label"] != "no_gesture"]

    # Get image dimensions
    dimension_rows = []

    for image_path in df["image_path"].unique():

        img = cv2.imread(image_path)

        if img is None:
            continue

        h, w = img.shape[:2]

        dimension_rows.append({
            "image_path": image_path,
            "width": w,
            "height": h,
            "width_height": f"{w} {h}"
        })

    df_dimensions = pd.DataFrame(dimension_rows)

    df = df.merge(
        df_dimensions,
        on="image_path",
        how="inner"
    )

    # Keep only 512x384 images if that is your final decision
    df = df[df["width_height"] == "512 384"]

    return df.reset_index(drop=True)

## 2. Applying transformations to the images

### 2.1. Transforming to gray scale and applying CLAHE

In [3]:
def preprocess_hand_crops(
    df,
    output_root_path,
    target_size=128,
    bbox_increase=0.10
):
    
    output_root_path = Path(output_root_path)
    output_root_path.mkdir(parents=True, exist_ok=True)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    processed_rows = []

    for _, row in df.iterrows():

        # Read original image
        img = cv2.imread(row["image_path"])

        if img is None:
            continue

        img_h, img_w = img.shape[:2]

        # Convert normalized bbox to pixel coordinates
        x1 = int(row["bbox_x"] * img_w)
        x2 = int((row["bbox_x"] + row["bbox_w"]) * img_w)

        y1 = int(row["bbox_y"] * img_h)
        y2 = int((row["bbox_y"] + row["bbox_h"]) * img_h)

        # Expand bbox
        bbox_w = x2 - x1
        bbox_h = y2 - y1

        margin_x = int(bbox_w * bbox_increase)
        margin_y = int(bbox_h * bbox_increase)

        x1 -= margin_x
        x2 += margin_x
        y1 -= margin_y
        y2 += margin_y

        # Clamp bbox to image limits
        x1 = max(0, x1)
        y1 = max(0, y1)

        x2 = min(img_w, x2)
        y2 = min(img_h, y2)

        # Crop hand
        hand_crop = img[y1:y2, x1:x2]

        if hand_crop.size == 0:
            continue

        # Convert crop to grayscale
        hand_crop = cv2.cvtColor(hand_crop, cv2.COLOR_BGR2GRAY)

        original_crop_h, original_crop_w = hand_crop.shape[:2]

        # Resize with padding
        h, w = hand_crop.shape[:2]

        scale = target_size / max(h, w)

        new_w = int(w * scale)
        new_h = int(h * scale)

        resized = cv2.resize(hand_crop, (new_w, new_h))

        canvas = np.zeros(
            (target_size, target_size),
            dtype=np.uint8
        )

        x_offset = (target_size - new_w) // 2
        y_offset = (target_size - new_h) // 2

        canvas[
            y_offset:y_offset + new_h,
            x_offset:x_offset + new_w
        ] = resized

        # Apply CLAHE once after crop + resize
        final_img = clahe.apply(canvas)

        # Save image
        label_dir = output_root_path / row["label"]
        label_dir.mkdir(parents=True, exist_ok=True)

        output_path = label_dir / f"{row['image_id']}.jpg"

        success = cv2.imwrite(str(output_path), final_img)

        if not success:
            continue

        processed_rows.append({
            "image_id": row["image_id"],
            "label": row["label"],
            "image_path": str(output_path),
            "original_crop_width": original_crop_w,
            "original_crop_height": original_crop_h,
            "original_crop_area": original_crop_w * original_crop_h,
            "final_width": target_size,
            "final_height": target_size,
            "final_resolution": f"{target_size}x{target_size}"
        })

    df_processed = pd.DataFrame(processed_rows)

    return df_processed

### 2.2. Getting hands from the bounding box and apply resizing

## 3. Complete Pipeline

In [4]:
def pipeline(input_image_path, input_annotations_path, output_image_path):

    df_loaded = load_data(
        input_image_path,
        input_annotations_path
    )

    df_processed = preprocess_hand_crops(
        df_loaded,
        output_image_path,
        target_size=128,
        bbox_increase=0.10
    )

    return df_loaded, df_processed

In [12]:
# Executing pipeline
input_image_path = "../data/processed/images"
input_annotations_path = "../data/processed/annotations"
output_image_path = "../data/processed/processed_images"
pipeline(input_image_path, input_annotations_path, output_image_path)

KeyError: 'image_exists'